# Chapitre 1 : structure de base d'un prompt

- [Cours](#cours)
- [Exercices](#exercices)
- [Espace de test](#espace-de-test)

## Configuration

Exécutez la cellule de configuration suivante pour charger votre clé API et définir la fonction utilitaire `get_completion`.

In [ ]:
!pip install anthropic

# Importer la bibliothèque standard de Python pour les expressions régulières
import re
import anthropic

# Récupérer les variables API_KEY et MODEL_NAME depuis le store IPython
%store -r API_KEY
%store -r MODEL_NAME

client = anthropic.Anthropic(api_key=API_KEY)

def get_completion(prompt: str, system_prompt=""):
    message = client.messages.create(
        model=MODEL_NAME,
        max_tokens=2000,
        temperature=0.0,
        system=system_prompt,
        messages=[
          {"role": "user", "content": prompt}
        ]
    )
    return message.content[0].text

---

## Cours

Anthropic propose deux API : l'ancienne [API Text Completions](https://docs.anthropic.com/claude/reference/complete_post) et l'actuelle [API Messages](https://docs.anthropic.com/claude/reference/messages_post). Dans ce tutoriel, nous utilisons exclusivement l'API Messages.

Au minimum, un appel à Claude via l'API Messages nécessite les paramètres suivants :
- `model` : le [nom API du modèle](https://docs.anthropic.com/claude/docs/models-overview#model-recommendations) que vous voulez appeler.

- `max_tokens` : le nombre maximal de tokens à générer avant l'arrêt. Claude peut s'arrêter avant d'atteindre ce maximum : ce paramètre ne fixe que la limite absolue. C'est aussi un arrêt *brutal* : Claude peut s'interrompre au milieu d'un mot ou d'une phrase.

- `messages` : un tableau de messages d'entrée. Nos modèles sont entraînés à fonctionner sur des tours de conversation alternés `user` et `assistant`. Lorsque vous créez un nouveau `Message`, vous indiquez les tours précédents dans le paramètre messages, et le modèle génère le `Message` suivant de la conversation.
  - Chaque message d'entrée doit être un objet comportant un `role` et un `content`. Vous pouvez fournir un seul message de rôle `user`, ou plusieurs messages `user` et `assistant` (qui doivent alors alterner). Le premier message doit toujours avoir le rôle `user`.

Il existe aussi des paramètres facultatifs, par exemple :
- `system` : le prompt système (voir plus bas).
  
- `temperature` : le degré de variabilité de la réponse de Claude. Pour ces leçons et exercices, `temperature` est fixée à 0.

Pour la liste complète des paramètres, consultez la [documentation de l'API](https://docs.anthropic.com/claude/reference/messages_post).

### Exemples

Voyons comment Claude répond à des prompts correctement formatés. Exécutez chacune des cellules suivantes (`Shift + Entrée`) : la réponse de Claude s'affiche sous le bloc.

In [ ]:
# Prompt
PROMPT = "Hi Claude, how are you?"

# Afficher la réponse de Claude
print(get_completion(PROMPT))

In [ ]:
# Prompt
PROMPT = "Can you tell me the color of the ocean?"

# Afficher la réponse de Claude
print(get_completion(PROMPT))

In [ ]:
# Prompt
PROMPT = "What year was Celine Dion born in?"

# Afficher la réponse de Claude
print(get_completion(PROMPT))

Voyons maintenant des prompts qui ne respectent pas le format de l'API Messages. Pour ces prompts mal formés, l'API renvoie une erreur.

Premier exemple : un appel à l'API Messages dont le tableau `messages` ne contient pas les champs `role` et `content`.

In [ ]:
# Obtenir la réponse de Claude
response = client.messages.create(
        model=MODEL_NAME,
        max_tokens=2000,
        temperature=0.0,
        messages=[
          {"Hi Claude, how are you?"}
        ]
    )

# Afficher la réponse de Claude
print(response[0].text)

Voici un prompt qui n'alterne pas les rôles `user` et `assistant`.

In [ ]:
# Obtenir la réponse de Claude
response = client.messages.create(
        model=MODEL_NAME,
        max_tokens=2000,
        temperature=0.0,
        messages=[
          {"role": "user", "content": "What year was Celine Dion born in?"},
          {"role": "user", "content": "Also, can you tell me some other facts about her?"}
        ]
    )

# Afficher la réponse de Claude
print(response[0].text)

Les messages `user` et `assistant` **DOIVENT alterner**, et la conversation **DOIT commencer par un tour `user`**. Un prompt peut contenir plusieurs paires `user` / `assistant` (pour simuler une conversation à plusieurs tours). Vous pouvez aussi placer des mots dans un dernier message `assistant` pour que Claude continue à partir de là (nous y reviendrons dans les chapitres suivants).

#### Prompts système

Vous pouvez aussi utiliser des **prompts système**. Un prompt système sert à **fournir à Claude du contexte, des instructions et des consignes** avant de lui soumettre une question ou une tâche dans le tour « User ».

Sur le plan de la structure, le prompt système est séparé de la liste des messages `user` et `assistant` : il se place dans un paramètre `system` distinct (regardez la structure de la fonction `get_completion` dans la section [Configuration](#configuration) du notebook).

Dans ce tutoriel, chaque fois qu'un prompt système peut servir, la fonction d'appel dispose d'un champ `system`. Si vous ne voulez pas de prompt système, donnez simplement la valeur chaîne vide à la variable `SYSTEM_PROMPT`.

#### Exemple de prompt système

In [ ]:
# Prompt système
SYSTEM_PROMPT = "Your answer should always be a series of critical thinking questions that further the conversation (do not provide answers to your questions). Do not actually answer the user question."

# Prompt
PROMPT = "Why is the sky blue?"

# Afficher la réponse de Claude
print(get_completion(PROMPT, SYSTEM_PROMPT))

Pourquoi utiliser un prompt système ? Un **prompt système bien rédigé améliore les performances de Claude** de plusieurs façons, par exemple en renforçant sa capacité à suivre des règles et des instructions. Pour en savoir plus, consultez notre documentation sur l'[utilisation des prompts système](https://docs.anthropic.com/claude/docs/how-to-use-system-prompts) avec Claude.

Passons aux exercices. Si vous voulez expérimenter avec les prompts du cours sans modifier le contenu ci-dessus, descendez tout en bas du notebook jusqu'à l'[**Espace de test**](#espace-de-test).

---

## Exercices
- [Exercice 1.1 - Compter jusqu'à trois](#exercice-11---compter-jusquà-trois)
- [Exercice 1.2 - Prompt système](#exercice-12---prompt-système)

### Exercice 1.1 - Compter jusqu'à trois
En respectant le format `user` / `assistant`, modifiez le `PROMPT` ci-dessous pour que Claude **compte jusqu'à trois**. Le résultat indique aussi si votre solution est correcte.

Note : le correcteur automatique vérifie la présence des chiffres 1, 2 et 3 dans la réponse, vous pouvez donc rédiger votre prompt en français.

In [ ]:
# Prompt : c'est le seul champ à modifier
PROMPT = "[Replace this text]"

# Obtenir la réponse de Claude
response = get_completion(PROMPT)

# Fonction de correction de l'exercice
def grade_exercise(text):
    pattern = re.compile(r'^(?=.*1)(?=.*2)(?=.*3).*$', re.DOTALL)
    return bool(pattern.match(text))

# Afficher la réponse de Claude et le résultat de la correction
print(response)
print("\n--------------------------- GRADING ---------------------------")
print("This exercise has been correctly solved:", grade_exercise(response))

❓ Pour obtenir un indice, exécutez la cellule ci-dessous.

In [ ]:
from hints import exercise_1_1_hint; print(exercise_1_1_hint)

### Exercice 1.2 - Prompt système

Modifiez le `SYSTEM_PROMPT` pour que Claude réponde comme un enfant de 3 ans.

Note : le correcteur cherche les mots anglais « giggles » ou « soo » dans la réponse. Gardez donc une réponse en anglais pour valider l'exercice, ou vérifiez le résultat vous-même.

In [ ]:
# Prompt système : c'est le seul champ à modifier
SYSTEM_PROMPT = "[Replace this text]"

# Prompt
PROMPT = "How big is the sky?"

# Obtenir la réponse de Claude
response = get_completion(PROMPT, SYSTEM_PROMPT)

# Fonction de correction de l'exercice
def grade_exercise(text):
    return bool(re.search(r"giggles", text) or re.search(r"soo", text))

# Afficher la réponse de Claude et le résultat de la correction
print(response)
print("\n--------------------------- GRADING ---------------------------")
print("This exercise has been correctly solved:", grade_exercise(response))

❓ Pour obtenir un indice, exécutez la cellule ci-dessous.

In [ ]:
from hints import exercise_1_2_hint; print(exercise_1_2_hint)

### Bravo !

Si vous avez résolu tous les exercices jusqu'ici, vous pouvez passer au chapitre suivant. Bon prompting !

---

## Espace de test

Cet espace vous permet d'expérimenter librement avec les exemples de prompts de la leçon et de les modifier pour voir l'effet sur les réponses de Claude.

In [ ]:
# Prompt
PROMPT = "Hi Claude, how are you?"

# Afficher la réponse de Claude
print(get_completion(PROMPT))

In [ ]:
# Prompt
PROMPT = "Can you tell me the color of the ocean?"

# Afficher la réponse de Claude
print(get_completion(PROMPT))

In [ ]:
# Prompt
PROMPT = "What year was Celine Dion born in?"

# Afficher la réponse de Claude
print(get_completion(PROMPT))

In [ ]:
# Obtenir la réponse de Claude
response = client.messages.create(
        model=MODEL_NAME,
        max_tokens=2000,
        temperature=0.0,
        messages=[
          {"Hi Claude, how are you?"}
        ]
    )

# Afficher la réponse de Claude
print(response[0].text)

In [ ]:
# Obtenir la réponse de Claude
response = client.messages.create(
        model=MODEL_NAME,
        max_tokens=2000,
        temperature=0.0,
        messages=[
          {"role": "user", "content": "What year was Celine Dion born in?"},
          {"role": "user", "content": "Also, can you tell me some other facts about her?"}
        ]
    )

# Afficher la réponse de Claude
print(response[0].text)

In [ ]:
# Prompt système
SYSTEM_PROMPT = "Your answer should always be a series of critical thinking questions that further the conversation (do not provide answers to your questions). Do not actually answer the user question."

# Prompt
PROMPT = "Why is the sky blue?"

# Afficher la réponse de Claude
print(get_completion(PROMPT, SYSTEM_PROMPT))